
# 🧮 Week 2 — How LLMs Work: Probability, Sampling & Prompts


<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/probability_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>


📘 **Theme:** Understanding LLMs as probabilistic systems — linking math intuition to observed behavior.

---

### **Learning Objectives**
By the end of this week, you will be able to:
1. Express the probability model underlying an LLM.
2. Explain the role of maximum likelihood in training.
3. Describe sampling, temperature, and entropy intuitively.
4. Analyze prompt conditioning effects on generation.
5. Connect mathematical modeling to user experience and reliability.


In [1]:
# @title Setup (Run this first)
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append('/content/main')
from course_utils import lab2_setup

lab2_setup()
print("✅ Environment ready!")

Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
✅ LAB2_colab_bootstrap complete — scientific libraries ready, helper function loaded.
✅ Environment ready!



## 🧩 Day 1 — A Simple Model of an LLM
---
**Guiding Question:**  
> What does it mean to say that an LLM “predicts the next token”?



### 🔢 LLM as a Probability Distribution

An LLM defines a joint probability over tokens:

$$
P(x_1, x_2, ..., x_T) = \prod_{t=1}^{T} P(x_t \mid x_{1:t-1})
$$

| Context | Next Token | Probability |
|----------|-------------|-------------|
| “The cat sat on the” | mat | 0.72 |
| | rug | 0.18 |
| | table | 0.10 |

Let's simulate next-token sampling.


In [2]:
import numpy as np

tokens = ["is", "was", "will be", "seems"]
probs = np.array([0.65, 0.20, 0.10, 0.05])

for _ in range(5):
    print("Next token:", np.random.choice(tokens, p=probs))

Next token: is
Next token: seems
Next token: was
Next token: is
Next token: is



### 🧮 Maximum Likelihood Training (Intuition)

The model learns parameters $\theta$ that maximize the likelihood of observed tokens:

$$
\theta^* = \arg\max_{\theta} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_{1:t-1})
$$

Taking the log converts products to sums, making optimization tractable.


In [3]:
true_next = "mat"
pred_probs = {"mat": 0.7, "rug": 0.2, "dog": 0.1}

import numpy as np
print("True token:", true_next)
print("Predicted probabilities:", pred_probs)
print("Log-likelihood contribution:", np.log(pred_probs[true_next]))

True token: mat
Predicted probabilities: {'mat': 0.7, 'rug': 0.2, 'dog': 0.1}
Log-likelihood contribution: -0.35667494393873245



### 🎭 Why This Causes Hallucinations

Because models maximize *probability of language*, not *truth*.

$$
P(\text{"2 + 2 = 4"}) \approx 0.99, \quad P(\text{"2 + 2 = 5"}) \approx 0.01
$$

For unseen contexts, they assign high probability to *plausible-sounding* continuations.

> Hallucination = high $P(\text{text}|\text{context}))$, low $P(\text{truth}|\text{world})$.



### 🧩 Unifying Diagram v1 — Adding Training Data Distribution

![Unifying Diagram v1](sandbox:/mnt/data/A_flowchart_diagram_illustrates_the_architecture_o.png)

Everything the model knows comes from its **training data distribution**, which shapes its probabilities.


In [4]:
# 🧠 Concept Check — Day 1
answer1 = "conditional probabilities"
answer2 = "maximize probability of observed data"

assert "conditional" in answer1.lower()
assert "probability" in answer2.lower()
print("✅ Correct! You understand Day 1 core concepts.")

✅ Correct! You understand Day 1 core concepts.



## 💻 Day 2 — Sampling, Temperature & Prompt Patterns
---
**Guiding Question:**  
> How does randomness shape creativity and control in LLMs?



### 🌡️ Sampling and Temperature

Softmax with temperature controls randomness:

$$
P_T(x_t=i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

- **Low T (e.g. 0.3)** → deterministic, repetitive  
- **High T (e.g. 2.0)** → diverse, creative


In [5]:
import numpy as np

def softmax_temp(z, T):
    p = np.exp(z / T)
    p /= p.sum()
    return p

logits = np.array([2.0, 1.0, 0.1])
for T in [0.5, 1.0, 2.0]:
    print(f'T={T}:', softmax_temp(logits, T))

T=0.5: [0.86377712 0.11689952 0.01932336]
T=1.0: [0.65900114 0.24243297 0.09856589]
T=2.0: [0.50168776 0.30428901 0.19402324]



### 🔀 Entropy: Measuring Uncertainty

Entropy quantifies uncertainty:

$$
H(P) = -\sum_i P_i \log_2 P_i
$$

- Low $H$: one outcome dominates (predictable).  
- High $H$: many outcomes possible (creative).


In [6]:
import math

def entropy(p): return -sum(pi * math.log(pi, 2) for pi in p)

print("Entropy([0.9, 0.1]) =", entropy([0.9, 0.1]))
print("Entropy([0.5, 0.5]) =", entropy([0.5, 0.5]))

Entropy([0.9, 0.1]) = 0.4689955935892812
Entropy([0.5, 0.5]) = 1.0



### 🧠 Prompt Patterns and Conditional Probability

Prompting changes the conditioning context:

$$
P_\theta(x_t \mid x_{1:t-1}, \text{prompt})
$$

**Common Patterns:**
1. *Role*: “You are a helpful tutor…”  
2. *Examples*: “Q: ... A: ...”  
3. *Constraints*: “Answer in 3 sentences.”



### 🧪 Bridge to Lab 2 — Temperature & Diversity

Students will vary `temperature` and measure output diversity:

$$
\text{Diversity Index} = \frac{\text{unique tokens}}{\text{total tokens}}
$$


In [7]:
# 🧠 Concept Check — Day 2
answer1 = "controls randomness"
answer2 = "higher"

assert "random" in answer1.lower()
assert "high" in answer2.lower()
print("✅ Excellent! Day 2 concepts mastered.")

✅ Excellent! Day 2 concepts mastered.



<details>
<summary>🧑‍🏫 Instructor Notes</summary>

**Day 1 Highlights:**
- Use simple numeric examples for probabilities.
- Avoid notation overload — emphasize meaning.

**Day 2 Highlights:**
- Show live sampling differences (temperature).
- Ask students: *“What kind of writing benefits from high entropy?”*
- Link back to Unifying Diagram: prompt → control layer.

**Extensions:**
- Optional mini-demo: `top_p` sampling vs temperature.
</details>
